# Matching Trailer to Success

In this notebook we e take the trailers collected in Notebook 02 and try to match them to movi s
based on their cleaned titles and release years.

The result is a combined dataset that contains both:
- movie success information (e.g. rating, revenue)
- trailer information (YouTube video ID)

This combined dataset is stored for further processing.

### Preparing helper functions
These helper functions clean and normalize titles and years and handle
ambiguous cases, making it possible to match trailers to movies
in a consistent and controlled way.

**`normalize_year_bytes`**

Movie years in the dataset are sometimes stored as strings like `"2009.0"`.
This function converts the raw HBase value into a clean integer year.
If the value is missing or cannot be parsed, it returns `None`.

**`base_title`**

Movie and trailer titles often differ slightly in formatting.
To make matching more robust, titles are reduced to a simplified form:

- everything is converted to lowercase
- subtitles after `:` are removed
- year information in brackets is removed
- extra whitespace and special characters are cleaned up

**`pick_the_best_candidate`**

In some cases, multiple movies may match the same cleaned trailer title.
This function resolves such conflicts:

- if a movie with the same release year exists, it is preferred
- otherwise, the movie with the highest number of votes is selected

In [1]:
import re

def normalize_year_bytes(b):
    if not b:
        return None
    s = b.decode("utf-8", errors="ignore").strip()
    if not s:
        return None
    # movie years are like "2009.0"
    try:
        return int(float(s))
    except:
        return None

def base_title(s: str) -> str:
    """
    Aggressiver Key fürs Matching:
    - lower
    - alles nach ':' weg
    - Klammern entfernen
    - Whitespace normalisieren
    """
    if not s:
        return ""
    t = s.lower()

    # alles nach ':' weg (häufig Untertitel)
    t = t.split(":", 1)[0]

    # Jahre in Klammern raus (sollten eh weg sein, aber sicher ist sicher)
    t = re.sub(r"\(\s*\d{4}\s*\)", "", t)

    # Sonderzeichen vereinheitlichen, Whitespace
    t = re.sub(r"[|–—]", "-", t)
    t = re.sub(r"\s+", " ", t).strip(" -:")

    return t

def pick_best_candidate(cands, target_year=None):
    """
    Falls mehrere Movie-Kandidaten:
    - bevorzugt exakt passendes Jahr
    - sonst Kandidat mit höchsten votes (wenn vorhanden)
    """
    if not cands:
        return None
    if target_year is not None:
        for c in cands:
            if c.get("year_int") == target_year:
                return c

    # fallback: max votes
    def votes_int(c):
        v = c.get("votes")
        try:
            return int(float(v))
        except:
            return 0

    return max(cands, key=votes_int)

### Preparing movie data for matching

In this step, we connect to HBase and make sure the table for the final result (`trailer_movie_join`) exists.
Then, all movie records are read once from the movie success table.

For each movie, the title and year are cleaned and normalized, and all relevant movie information
(rating, revenue, votes, success flag) is collected.
Based on this, several in-memory lookup structures are built using different combinations of
title a if needed.


In [2]:
import happybase
from collections import defaultdict

HBASE_HOST = "hbase"
HBASE_PORT = 9090

MOVIE_TABLE = "movie_success_rate_raw"
VIDEO_TABLE = "kinocheck_videos"
JOIN_TABLE  = "trailer_movie_join"

# ---------- Connect + Ensure join table ----------

conn = happybase.Connection(host=HBASE_HOST, port=HBASE_PORT, timeout=20000)
conn.open()

if JOIN_TABLE.encode() not in conn.tables():
    conn.create_table(JOIN_TABLE, {"cf": dict()})

movie_table = conn.table(MOVIE_TABLE)
video_table = conn.table(VIDEO_TABLE)
join_table  = conn.table(JOIN_TABLE)

# ---------- Build Movie Indexes ----------
# title_norm -> list[candidate]
# (title_norm, year) -> list[candidate]
# base_title -> list[candidate]
# (base_title, year) -> list[candidate]

idx_title = defaultdict(list)
idx_title_year = defaultdict(list)
idx_base = defaultdict(list)
idx_base_year = defaultdict(list)

for rk, data in movie_table.scan(columns=[b"cf:title_norm", b"cf:year", b"cf:title", b"cf:success", b"cf:rating", b"cf:revenue", b"cf:votes"]):
    title_norm = data.get(b"cf:title_norm", b"").decode("utf-8", errors="ignore").strip()
    if not title_norm:
        # falls du cf:title_norm noch nicht gesetzt hast, kannst du hier notfalls cf:title normalisieren
        title_raw = data.get(b"cf:title", b"").decode("utf-8", errors="ignore").strip()
        title_norm = title_raw.lower()

    year_int = normalize_year_bytes(data.get(b"cf:year"))
    btitle = base_title(title_norm)

    cand = {
        "rowkey": rk,
        "title": data.get(b"cf:title", b"").decode("utf-8", errors="ignore"),
        "title_norm": title_norm,
        "base": btitle,
        "year_int": year_int,
        "year_raw": data.get(b"cf:year", b"").decode("utf-8", errors="ignore"),
        "success": data.get(b"cf:success", b"").decode("utf-8", errors="ignore"),
        "rating": data.get(b"cf:rating", b"").decode("utf-8", errors="ignore"),
        "revenue": data.get(b"cf:revenue", b"").decode("utf-8", errors="ignore"),
        "votes": data.get(b"cf:votes", b"").decode("utf-8", errors="ignore"),
    }

    idx_title[title_norm].append(cand)
    if year_int is not None:
        idx_title_year[(title_norm, year_int)].append(cand)

    idx_base[btitle].append(cand)
    if year_int is not None:
        idx_base_year[(btitle, year_int)].append(cand)

print("Movie Index Keys (title_norm):", len(idx_title))
print("Movie Index Keys (base):", len(idx_base))


Movie Index Keys (title_norm): 837
Movie Index Keys (base): 806


### Matching trailers to movies and writing the result

In this step, we go through all trailers stored in `kinocheck_videos` and try to link each one to a movie from the success dataset.

For every trailer, we first skip anything marked as non-trailer (`skip=true`) and ignore entries without a cleaned title.  
Then we read the trailer year (if available) and build two matching keys:
- the exact cleaned title (`tnorm`)
- a simplified “base title” (`tbase`) that removes things like subtitles and formatting differences

Matching is done in the following order:
1. exact title + year  
2. base title + year  
3. exact title only  
4. base title only  

If multiple movies match, we pick the best candidate (prefer same year, otherwise highest votes).
We track basic stats (matched, unmatched, and which strategy worked), so we can later explain how good the join was.

All successful matches are then written into the HBase table `trailer_movie_join`.
Each row is keyed by the `video_id` and contains both trailer fields and the movie fields (rating, revenue, success, votes), plus the `match_type` so we know how the match was made.


In [3]:
# ---------- Match trailers ----------
matches = []
stats = defaultdict(int)

for rk, data in video_table.scan(columns=[b"yt:title_clean", b"yt:year", b"yt:skip"]):
    if data.get(b"yt:skip") == b"true":
        stats["skipped_trailers"] += 1
        continue

    tclean = data.get(b"yt:title_clean", b"").decode("utf-8", errors="ignore").strip()
    if not tclean:
        stats["no_title_clean"] += 1
        continue

    tyear = None
    yb = data.get(b"yt:year", b"")
    if yb:
        try:
            tyear = int(yb.decode("utf-8", errors="ignore").strip())
        except:
            tyear = None

    tnorm = tclean.lower()
    tbase = base_title(tnorm)

    chosen = None
    match_type = None

    # 1) EXACT_TITLE_YEAR
    if tyear is not None:
        cands = idx_title_year.get((tnorm, tyear), [])
        chosen = pick_best_candidate(cands, tyear)
        if chosen:
            match_type = "exact_title_year"

    # 2) BASE_TITLE_YEAR
    if not chosen and tyear is not None:
        cands = idx_base_year.get((tbase, tyear), [])
        chosen = pick_best_candidate(cands, tyear)
        if chosen:
            match_type = "base_title_year"

    # 3) EXACT_TITLE_ONLY
    if not chosen:
        cands = idx_title.get(tnorm, [])
        chosen = pick_best_candidate(cands, tyear)
        if chosen:
            match_type = "exact_title_only"

    # 4) BASE_TITLE_ONLY
    if not chosen:
        cands = idx_base.get(tbase, [])
        chosen = pick_best_candidate(cands, tyear)
        if chosen:
            match_type = "base_title_only"

    if not chosen:
        stats["unmatched"] += 1
        continue

    stats["matched"] += 1
    stats[f"matched_{match_type}"] += 1

    video_id = rk.decode("utf-8", errors="ignore")

    matches.append({
        "video_id": video_id,
        "trailer_title_clean": tclean,
        "trailer_year": "" if tyear is None else str(tyear),
        "movie_rowkey": chosen["rowkey"],
        "movie_title": chosen["title"],
        "movie_year": chosen["year_raw"],
        "success": chosen["success"],
        "rating": chosen["rating"],
        "revenue": chosen["revenue"],
        "votes": chosen["votes"],
        "match_type": match_type,
    })

print("Matches:", stats["matched"])
print("Unmatched:", stats["unmatched"])
print("Breakdown:", {k: v for k, v in stats.items() if k.startswith("matched_")})

# ---------- Write joins to HBase ----------
with join_table.batch(batch_size=200) as b:
    for m in matches:
        rowkey = m["video_id"].encode("utf-8")
        b.put(rowkey, {
            b"cf:video_id": m["video_id"].encode("utf-8"),
            b"cf:trailer_title_clean": m["trailer_title_clean"].encode("utf-8", errors="ignore"),
            b"cf:trailer_year": m["trailer_year"].encode("utf-8"),
            b"cf:movie_rowkey": m["movie_rowkey"],  # bytes
            b"cf:movie_title": m["movie_title"].encode("utf-8", errors="ignore"),
            b"cf:movie_year": m["movie_year"].encode("utf-8", errors="ignore"),
            b"cf:success": m["success"].encode("utf-8", errors="ignore"),
            b"cf:rating": m["rating"].encode("utf-8", errors="ignore"),
            b"cf:revenue": m["revenue"].encode("utf-8", errors="ignore"),
            b"cf:votes": m["votes"].encode("utf-8", errors="ignore"),
            b"cf:match_type": m["match_type"].encode("utf-8"),
        })

print("Join rows written:", len(matches))
conn.close()

Matches: 249
Unmatched: 250
Breakdown: {'matched_exact_title_year': 204, 'matched_exact_title_only': 29, 'matched_base_title_year': 14, 'matched_base_title_only': 2}
Join rows written: 249


### Verifying the matched trailer movie data

This cell reads a small sample from the `trailer_movie_join` table and prints it.
It is used to check to see whether trailers were linked to the correct movies and whether the stored fields (title, year, rating, match type) look reasonable.


In [4]:
import happybase

HBASE_HOST = "hbase"
HBASE_PORT = 9090
JOIN_TABLE = "trailer_movie_join"

conn = happybase.Connection(host=HBASE_HOST, port=HBASE_PORT)
conn.open()

table = conn.table(JOIN_TABLE)

print("=== Stichprobe: trailer_movie_join (10 Rows) ===")
i = 0
for rk, data in table.scan(
    limit=10,
    columns=[
        b"cf:trailer_title_clean",
        b"cf:trailer_year",
        b"cf:movie_title",
        b"cf:movie_year",
        b"cf:match_type",
        b"cf:success",
        b"cf:rating",
        b"cf:revenue",
    ]
):
    print(
        f"{i+1:02d}. video_id={rk.decode('utf-8', errors='ignore')}",
        "| trailer =", data.get(b"cf:trailer_title_clean", b"").decode("utf-8", errors="ignore"),
        "| trailer_year =", data.get(b"cf:trailer_year", b"").decode("utf-8", errors="ignore"),
        "| movie =", data.get(b"cf:movie_title", b"").decode("utf-8", errors="ignore"),
        "| movie_year =", data.get(b"cf:movie_year", b"").decode("utf-8", errors="ignore"),
        "| match_type =", data.get(b"cf:match_type", b"").decode("utf-8", errors="ignore"),
        "| rating =", data.get(b"cf:rating", b"").decode("utf-8", errors="ignore"),
    )
    i += 1

conn.close()


=== Stichprobe: trailer_movie_join (10 Rows) ===
01. video_id=-GQjm4TrQIM | trailer = alice through the looking glass | trailer_year = 2016 | movie = Alice Through the Looking Glass | movie_year = 2016.0 | match_type = exact_title_year | rating = 6.2
02. video_id=-gD-ywBXQns | trailer = the last witch hunter | trailer_year = 2015 | movie = The Last Witch Hunter | movie_year = 2015.0 | match_type = exact_title_year | rating = 6.0
03. video_id=0A5rvlALfBk | trailer = the huntsman: winter's war | trailer_year = 2016 | movie = The Huntsman: Winter's War | movie_year = 2016.0 | match_type = exact_title_year | rating = 6.1
04. video_id=16jSEcCTO8o | trailer = black mass | trailer_year = 2015 | movie = Black Mass | movie_year = 2015.0 | match_type = exact_title_year | rating = 6.9
05. video_id=17Mj1VQBvss | trailer = finding dory | trailer_year = 2016 | movie = Finding Dory | movie_year = 2016.0 | match_type = exact_title_year | rating = 7.4
06. video_id=1FuH-Oc6N-w | trailer = the shallows |